# Week 3 — prepare the data and establish the K-means baseline

    This is the simplified analysis. Read the cells in order. Each section states
    its purpose. Models are declared directly with scikit-learn or SciPy.

    The scientific design, country split, cluster IDs and frozen profile definitions have
    not changed. The notebook reads the completed, saved experiments by default.
    Expensive refits are disabled explicitly. Refit outputs go to a new folder;
    they never overwrite the original evidence or the Week 6 input files.
    Optional refits produce comparison files; they do not replace frozen labels
    or automatically replace the displayed historical summaries.

    The original notebook (historical notebook retained in the course archive) preserves the full discussion,
    run history, figures, computational failures and implementation details.
    This rewrite is not a new model-selection exercise.


Second readability pass: smaller cells, imports in one place, and explicit method functions.

## Start here

1. Keep the course folder structure: `weekly_notebooks`, `23_wrp` and
   `course_artifacts` next to each other. These notebooks are not data downloads.
2. Use the packages in `requirements-objective1.txt`. The recorded versions are
   the ones used for the verification. Start a fresh Python kernel, then run
   all cells from top to bottom. You do not need to run another notebook first.
3. Leave the refit switches below at `False` to reproduce the saved analysis
   without launching the expensive searches. Cells still recalculate the checks
   and summaries shown in their code. A switch set to `True` writes new comparison
   outputs under `course_artifacts/objective1_simplified_checks`; it does not
   replace the frozen results.

**What is needed to reproduce this:** the 2023 source data, frozen country split,
prepared datasets, and saved model/experiment files referenced below. These are
deliberate scientific inputs. The notebooks do not reconstruct every historical
experiment from the raw CSV alone. The original full searches remain in the
linked archived notebooks. Only load the trusted local course `.joblib` files.

The order is: **Week 3 baseline → Week 4 K-means → Week 4 alternative methods →
Week 4 partition comparisons → Week 5 validation.** All saved inputs are unchanged.

### Packages

With Python 3.11, install once from the course folder:
`python -m pip install -r weekly_notebooks/requirements-objective1.txt`

In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import cdist
from threadpoolctl import threadpool_limits
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, f1_score


### Optional refits — off by default

In [ ]:
RUN_BASELINE = False       # 9 full fits, plus 36 initialization-stability fits.
RUN_PREDICTION_STRENGTH = False  # 360 fits; not needed to inspect completed results.
RUN_EXPLANATIONS = False


## 1. Recode the 13 profile variables and reuse the frozen country split

    Worry: 0 = not worried, 1 = somewhat worried, 2 = very worried.
    Harm: 0 = neither affected; 1 = respondent, someone known, or both affected.
    Codes 98/99 remain missing. Keep all source columns and all respondents.
    The split is an input: do not search for another country allocation.


In [ ]:
# Run from the course folder or weekly_notebooks. No absolute data paths.
ROOT = Path.cwd()
while not (ROOT / 'course_artifacts').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'course_artifacts').is_dir(), 'Open the course folder first.'
ART = ROOT / 'course_artifacts'
OUT = ART / 'objective1_simplified_checks'
OUT.mkdir(exist_ok=True)


In [ ]:
DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + d for d in DOMAINS]
HARM = ['harm_' + d for d in DOMAINS + ['work']]


In [ ]:
def country_weights(frame):
    # Each country has equal total mass; preserve relative survey weights within it.
    return (frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
            / frame.COUNTRY_ISO3.nunique()).to_numpy()


In [ ]:
raw = pd.read_csv(ROOT / '23_wrp/WRP_2023.csv', low_memory=False)
prepared = raw.copy()
worry_sources = ['WP20720', 'WP20721', 'WP20722', 'WP20723', 'WP22213', 'WP20726']
harm_sources = ['WP22442', 'WP22443', 'WP22444', 'WP22445', 'WP22446', 'WP22447', 'WP22448']
for source, feature in zip(worry_sources, WORRY):
    values = pd.to_numeric(raw[source], errors='raise')
    assert values.dropna().isin([1, 2, 3, 98, 99]).all(), source
    prepared[feature] = values.map({1: 2., 2: 1., 3: 0.})
for source, feature in zip(harm_sources, HARM):
    values = pd.to_numeric(raw[source], errors='raise')
    assert values.dropna().isin([1, 2, 3, 4, 98, 99]).all(), source
    prepared[feature] = values.map({1: 1., 2: 1., 3: 1., 4: 0.})


In [ ]:
split = pd.read_csv(ART / 'week03_country_split.csv')
assert split.COUNTRY_ISO3.is_unique
prepared = prepared.merge(split[['COUNTRY_ISO3', 'country_split']],
                          on='COUNTRY_ISO3', how='left', validate='many_to_one')
assert prepared.WPID_RANDOM.is_unique and prepared.country_split.notna().all()
assert np.isfinite(prepared.WGT).all() and prepared.WGT.gt(0).all()
prepared['weight_equal_country'] = np.nan
for _, group in prepared.groupby('country_split'):
    prepared.loc[group.index, 'weight_equal_country'] = country_weights(group)
display(prepared.groupby('country_split').agg(respondents=('WPID_RANDOM', 'size'),
                                             countries=('COUNTRY_ISO3', 'nunique')))
# Verify the recoding and weights; do not overwrite the frozen prepared datasets.
saved = pd.read_parquet(ART / 'week03_2023_analysis_data.parquet').set_index('WPID_RANDOM')
check = prepared.set_index('WPID_RANDOM').loc[saved.index]
assert check.country_split.equals(saved.country_split)
assert np.allclose(check[WORRY + HARM], saved[WORRY + HARM], equal_nan=True)
assert np.allclose(check.weight_equal_country, saved.weight_equal_country)


## 2. Check missingness and fit preprocessing on development countries only

    Exclude respondents with no observed profile feature from clustering, but
    retain them in the frozen dataset. The baseline uses a global development
    median for each feature, not a separate median for each country. Imputation
    and z-standardization are unweighted. No PCA is used.

    Keep the original Week 3 weights after excluding all-missing rows. Week 4
    instead recalculates weights within its complete-response population. These
    are different historical designs, not interchangeable implementations.


In [ ]:
development = prepared.loc[prepared.country_split.eq('train')].copy()
profile = WORRY + HARM
display(development[profile].isna().mean().rename('missing_fraction'))
display(development[profile].isna().sum(axis=1).value_counts().sort_index())
country_missing = development[profile].isna().groupby(development.COUNTRY_ISO3).mean()
display(country_missing)
all_missing = development[profile].isna().all(axis=1)
print('Development respondents with no observed profile value:', int(all_missing.sum()))
data = development.loc[~all_missing].reset_index(drop=True)
weights = data.weight_equal_country.to_numpy()
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X = scaler.fit_transform(imputer.fit_transform(data[profile]))
quality_rows = np.random.default_rng(2023).choice(len(data), min(10000, len(data)), replace=False)


In [ ]:
# Saved centroids are in original survey units. Recover the baseline labels.
centroids = pd.read_csv(ART / 'week03_baseline_kmeans_centroids.csv')
baseline_labels = {}
for k in range(2, 11):
    centres = centroids.loc[centroids.k.eq(k)].sort_values('cluster')[profile]
    baseline_labels[k] = cdist(X, scaler.transform(centres)).argmin(axis=1)
metrics = pd.read_csv(ART / 'week03_baseline_kmeans_metrics.csv')


## 3. Fit weighted K-means for k = 2,...,10

    Use 30 initializations and seed 2023. Check initialization stability with
    four independent seeds, each with five initializations. Internal quality
    measures use the same fixed sample of 10,000 respondents for every k.
    These quality measures and ARI are unweighted; the K-means fit uses weights.
    The size-imbalance statistic is half the sum of deviations from equal shares.


In [ ]:
if RUN_BASELINE:
    rows, size_rows = [], []
    total = np.sum(weights[:, None] * (X - np.average(X, axis=0, weights=weights)) ** 2)
    for k in range(2, 11):
        with threadpool_limits(limits=1):
            model = KMeans(n_clusters=k, n_init=30, random_state=2023)
            labels = model.fit_predict(X, sample_weight=weights)
            repeated = [KMeans(n_clusters=k, n_init=5, random_state=seed)
                        .fit_predict(X, sample_weight=weights) for seed in [11, 29, 47, 71]]
        baseline_labels[k] = labels
        shares = np.bincount(labels, weights=weights, minlength=k) / weights.sum()
        respondent_shares = np.bincount(labels, minlength=k) / len(labels)
        size_rows.extend({'k': k, 'cluster': j, 'respondents': int((labels == j).sum()),
                          'respondent_pct': 100 * respondent_shares[j],
                          'equal_country_weighted_pct': 100 * shares[j]} for j in range(k))
        rows.append({'k': k, 'inertia': model.inertia_,
                     'explained_dispersion_pct': 100 * (1 - model.inertia_ / total),
                     'cluster_size_imbalance_pct': 50 * np.abs(shares - 1 / k).sum(),
                     'max_weighting_share_gap_pct_points': 100 * np.max(np.abs(shares - respondent_shares)),
                     'silhouette': silhouette_score(X[quality_rows], labels[quality_rows]),
                     'calinski_harabasz': calinski_harabasz_score(X[quality_rows], labels[quality_rows]),
                     'davies_bouldin': davies_bouldin_score(X[quality_rows], labels[quality_rows]),
                     'initialization_stability_ari': np.mean([
                         adjusted_rand_score(a, b) for a, b in combinations(repeated, 2)])})
    metrics = pd.DataFrame(rows)
    metrics['incremental_inertia_improvement_pct'] = -100 * metrics.inertia.pct_change()
    metrics.to_csv(OUT / 'week03_refitted_baseline_metrics.csv', index=False)
    pd.DataFrame(size_rows).to_csv(OUT / 'week03_refitted_sizes.csv', index=False)
display(metrics.round(4))


## 4. Repeated country-block Prediction Strength

    Ten random country-half splits, evaluated in both directions, give 20 scores
    per k. Learn the median and scaler on the training countries. Independently
    cluster the other half in that same training-fitted feature geometry.
    For each independent test cluster, count the fraction of pairs kept together
    by the transported model. Prediction Strength is the minimum across clusters.
    Retain every k with mean score at least 0.75. SD is not a confidence interval.


In [ ]:
def prediction_strength(independent, transported, k):
    scores = []
    for cluster in range(k):
        members = independent == cluster
        n = members.sum()
        counts = np.bincount(transported[members], minlength=k)
        scores.append(np.sum(counts * (counts - 1)) / (n * (n - 1)) if n > 1 else 0.)
    return min(scores)


In [ ]:
if RUN_PREDICTION_STRENGTH:
    rows = []
    rng = np.random.default_rng(2023)
    countries = np.array(sorted(data.COUNTRY_ISO3.unique()))
    for repeat in range(10):
        shuffled = rng.permutation(countries)
        a, b = shuffled[:len(countries) // 2], shuffled[len(countries) // 2:]
        for direction, train_countries, test_countries in [(1, a, b), (2, b, a)]:
            train = data.COUNTRY_ISO3.isin(train_countries).to_numpy()
            test = data.COUNTRY_ISO3.isin(test_countries).to_numpy()
            fold_imputer, fold_scaler = SimpleImputer(strategy='median'), StandardScaler()
            x_train = fold_scaler.fit_transform(fold_imputer.fit_transform(data.loc[train, profile]))
            x_test = fold_scaler.transform(fold_imputer.transform(data.loc[test, profile]))
            for k in range(2, 11):
                seed = 2023 + 1000 * repeat + 100 * k + direction
                with threadpool_limits(limits=1):
                    train_model = KMeans(n_clusters=k, n_init=10, random_state=seed)
                    train_model.fit(x_train, sample_weight=weights[train])
                    independent = KMeans(n_clusters=k, n_init=10, random_state=seed + 50000)
                    independent_labels = independent.fit_predict(x_test, sample_weight=weights[test])
                rows.append({'repeat': repeat + 1, 'direction': direction, 'k': k,
                             'prediction_strength': prediction_strength(
                                 independent_labels, train_model.predict(x_test), k)})
    folds = pd.DataFrame(rows)
    folds.to_csv(OUT / 'week03_refitted_prediction_strength.csv', index=False)
else:
    folds = pd.read_csv(ART / 'week03_prediction_strength_folds.csv')
ps = folds.groupby('k').prediction_strength.agg(['mean', 'std', 'min', 'max', 'count'])
ps['fraction_at_least_075'] = folds.assign(passes=folds.prediction_strength.ge(.75)).groupby('k').passes.mean()
ps['retained'] = ps['mean'].ge(.75)
selected = ps.index[ps.retained].tolist()
display(ps.round(3))
assert selected == [2, 3, 4], 'The historical selection changed: inspect before proceeding.'
ps['mean'].plot(marker='o', ylim=(0, 1), ylabel='Mean Prediction Strength', title='Week 3 country-block reproducibility')
plt.axhline(.75, color='black', linestyle='--')
plt.show()


## 5. Explain the fixed clusters without refitting them

    Two checks: (1) predict cluster membership from overall worry and overall
    combined harm using depth-3/5 trees; (2) find the best single-feature binary
    rule for each cluster. Both use all five country-grouped CV folds.
    The single-feature search uses observed responses only and maximizes weighted
    balanced accuracy on the training countries. It is not ordinary Gini splitting.
    These are explanatory checks, not held-out profile-validation experiments.


In [ ]:
# The burden-tree function below uses the original Week 3 weights in each fold.
def burden_folds(frame, labels, weights, worry, harm, depths, seed):
    scores = pd.DataFrame({'worry': frame[worry].mean(axis=1),
                           'harm': frame[harm].mean(axis=1)})
    rows = []
    cv = GroupKFold(n_splits=5)
    for fold, (train, test) in enumerate(cv.split(scores, groups=frame.COUNTRY_ISO3), 1):
        train_weight = weights[train]
        test_weight = weights[test]
        imputer = SimpleImputer(strategy='median')
        x_train = imputer.fit_transform(scores.iloc[train])
        x_test = imputer.transform(scores.iloc[test])
        for depth in depths:
            tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=200,
                                          random_state=seed)
            tree.fit(x_train, labels[train], sample_weight=train_weight)
            prediction = tree.predict(x_test)
            rows.append({'fold': fold, 'depth': depth,
                         'balanced_accuracy': balanced_accuracy_score(
                             labels[test], prediction, sample_weight=test_weight),
                         'macro_f1': f1_score(labels[test], prediction, average='macro',
                                              sample_weight=test_weight, zero_division=0)})
    return pd.DataFrame(rows)


In [ ]:
def one_feature_folds(frame, labels, weights, features):
    # Search balanced-accuracy rules, not a Gini-optimized sklearn stump.
    rows = []
    cv = GroupKFold(n_splits=5)
    for fold, (train, test) in enumerate(cv.split(frame, groups=frame.COUNTRY_ISO3), 1):
        for cluster in np.unique(labels):
            target = labels == cluster
            best_score, best_rule = -np.inf, None
            for feature in features:
                x = frame[feature].to_numpy(float)
                observed = train[np.isfinite(x[train])]
                categories = np.unique(x[observed])
                if len(categories) < 2 or np.unique(target[observed]).size < 2:
                    continue
                for threshold in (categories[:-1] + categories[1:]) / 2:
                    for greater in [True, False]:
                        prediction = x[observed] > threshold if greater else x[observed] <= threshold
                        score = balanced_accuracy_score(target[observed], prediction,
                                                        sample_weight=weights[observed])
                        if score > best_score:
                            best_score, best_rule = score, (feature, threshold, greater)
            if best_rule is None:
                continue
            feature, threshold, greater = best_rule
            x = frame[feature].to_numpy(float)
            observed = test[np.isfinite(x[test])]
            prediction = x[observed] > threshold if greater else x[observed] <= threshold
            rows.append({'fold': fold, 'cluster': int(cluster), 'feature': feature,
                         'threshold': threshold, 'greater': greater,
                         'balanced_accuracy': balanced_accuracy_score(
                             target[observed], prediction, sample_weight=weights[observed]),
                         'precision': precision_score(target[observed], prediction,
                             sample_weight=weights[observed], zero_division=np.nan),
                         'recall': recall_score(target[observed], prediction,
                             sample_weight=weights[observed], zero_division=np.nan),
                         'observed_weight_pct': 100 * weights[observed].sum() / weights[test].sum()})
    return pd.DataFrame(rows)


In [ ]:
if RUN_EXPLANATIONS:
    for k in selected:
        burden_folds(data, baseline_labels[k], weights, WORRY, HARM, [3, 5], 2023).to_csv(
            OUT / f'week03_burden_k{k}.csv', index=False)
        one_feature_folds(data, baseline_labels[k], weights, profile).to_csv(
            OUT / f'week03_one_feature_k{k}.csv', index=False)
display(pd.read_csv(ART / 'week03_burden_fidelity_summary.csv').round(3))
display(pd.read_csv(ART / 'week03_one_feature_summary.csv').round(3))
display(pd.read_csv(ART / 'week03_selected_k_distribution_profiles.csv').head(30))


## Conclusion

    Mean Prediction Strength retains k = 2, 3 and 4. This does not establish final
    profile definitions. The k = 4 water-harm cluster is nearly reconstructed by
    one feature (98.8% balanced accuracy, 85.2% precision, 100% recall). High
    initialization stability alone does not rule out a simple burden partition.
    Week 4 therefore tests feature representation, geometry and clustering method.


## Reproduction

The baseline fits and diagnostics are read from the completed Week 3 experiments. Optional fits never overwrite the frozen T5 model.

## Readability revision checked

All displayed tables, plots and printed results match the preceding version exactly. The revised GMM/DBSCAN functions also reproduced the tested reference labels exactly (ARI 1.000). No full stability search was repeated. See the [readability check](../course_artifacts/objective1_simplified_checks/readability_pass_report.json).